In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import os
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.10.0


In [ ]:

# Define paths
train_path = r'D:\AI_Projects\EIS\Data_Sets\train'
test_path = r'D:\AI_Projects\EIS\Data_Sets\test'

# Emotion labels
emotion_labels = {
    0: 'Angry',
    1: 'Disgust',
    2: 'Fear',
    3: 'Happy',
    4: 'Sad',
    5: 'Surprise',
    6: 'Neutral'
}

def load_dataset(dataset_path):
    images = []
    labels = []

    for emotion_num, emotion_name in emotion_labels.items():
        emotion_path = os.path.join(dataset_path, emotion_name)

        # Check if directory exists
        if not os.path.isdir(emotion_path):
            print(f"Warning: Directory {emotion_path} not found. Skipping...")
            continue

        for img_file in os.listdir(emotion_path):
            if img_file.endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(emotion_path, img_file)

                try:
                    # Read and preprocess image
                    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                    img = cv2.resize(img, (48, 48))

                    # Normalize pixel values
                    img = img / 255.0

                    images.append(img)
                    labels.append(emotion_num)
                except Exception as e:
                    print(f"Error loading image {img_path}: {e}")

    # Convert to numpy arrays
    images = np.array(images)
    labels = np.array(labels)

    return images, labels
# Load training and test data
print("Loading training data...")
train_images, train_labels = load_dataset(train_path)
print("Loading test data...")
test_images, test_labels = load_dataset(test_path)

# Reshape images for CNN input
train_images = train_images.reshape(-1, 48, 48, 1)
test_images = test_images.reshape(-1, 48, 48, 1)

# One-hot encode labels
train_labels = to_categorical(train_labels, num_classes=7)
test_labels = to_categorical(test_labels, num_classes=7)

# Split into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    train_images,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels  # Maintain class distribution
)

# Dataset statistics
print("\nDataset Statistics:")
print(f"Training set shape: {X_train.shape} ({(X_train.shape[0]/len(train_images)*100):.1f}% of total)")
print(f"Validation set shape: {X_val.shape} ({(X_val.shape[0]/len(train_images)*100):.1f}% of total)")
print(f"Test set shape: {test_images.shape}")

# Visualize class distribution
def plot_class_distribution(labels, title):
    class_counts = np.sum(labels, axis=0)
    plt.figure(figsize=(10, 5))
    plt.bar(list(emotion_labels.values()), class_counts,  tick_label=list(emotion_labels.values()))
    plt.title(title)
    plt.xlabel('Emotion Class')
    plt.ylabel('Number of Images')
    plt.xticks(rotation=45)
    plt.show()

plot_class_distribution(train_labels, 'Training Set Class Distribution')
plot_class_distribution(test_labels, 'Test Set Class Distribution')

Loading training data...
Loading test data...

Dataset Statistics:
Training set shape: (22967, 48, 48, 1) (80.0% of total)
Validation set shape: (5742, 48, 48, 1) (20.0% of total)
Test set shape: (7178, 48, 48, 1)


AttributeError: 'RcParams' object has no attribute '_get'